# EE473 Final Project: Energy-Aware Edge Scheduling with Reinforcement Learning

This notebook reports a reproducible RL scheduling study on real Google cluster traces.
A single device selects `low/medium/high` modes to balance energy and queueing quality.
All main figures/tables are loaded from saved artifacts under `results/`.


## 1) Problem Setup

### Environment (custom simulator)

| Component | Definition |
|---|---|
| State | `(workload_bin, queue_bin, battery_bin)` |
| Action | `low / medium / high` |
| Transition | workload arrival -> service -> queue/battery update |
| Termination | 288 steps (or optional battery cutoff) |

### Core constants

- Service rates: `(0.35, 0.75, 1.20)`
- Energy costs: `(0.18, 0.42, 0.85)`
- Queue cap: `8.0`, deadline queue threshold: `2.5`
- Switch cost: `0.05`

### Reward

`r_t = -(alpha_energy * energy_t + beta_latency * latency_t + gamma_miss * miss_t)`

Default weights: `(alpha, beta, gamma) = (1.0, 0.6, 2.0)`.


## 2) Dataset, Protocol, and Course Alignment

### Dataset and split

- Source: Google cluster task-event trace (120 shards)
- Preprocess: 60-second buckets, normalized workload, fixed-length episodes
- Episode length: `288`
- Split used in main protocol: train/test = `20/6` episodes (non-overlap)

### Methods compared

- Baselines: `always_low`, `always_medium`, `always_high`, `threshold`
- RL: tabular Q-learning and linear approximation Q-learning
- Reproducibility: 5 seeds with mean/std reporting

### Why this matches the course project track

The dataset project track requires RL on a moderately large dataset with at least two class methods;
this notebook compares tabular and function-approximation Q-learning under the same protocol.


## 3) Environment Walkthrough

This section loads the trace and runs one episode to verify the simulator interface (`reset/step`, state, reward, done, metrics info).


In [ ]:
from pathlib import Path
import subprocess
import json

ROOT = Path.cwd().resolve().parent
assert (ROOT / 'scripts').exists(), f'Expected repo root at {ROOT}'
print('ROOT =', ROOT)

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Add src/ to path so we can import project modules directly
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from config import DEFAULT_ENV_CONFIG
from data_loader import load_workload_trace, build_episodes
from env import EnergySchedulingEnv

# ── Load workload trace ──────────────────────────────────────────────────────
TRACE_PATH = ROOT / 'data/processed/workload_trace_120.csv'
EPISODE_LENGTH = 288
STRIDE = 288

train_series = load_workload_trace(TRACE_PATH, split='train')
test_series  = load_workload_trace(TRACE_PATH, split='test')

train_episodes = build_episodes(train_series, EPISODE_LENGTH, stride=STRIDE)[:20]
test_episodes  = build_episodes(test_series,  EPISODE_LENGTH, stride=STRIDE)[:6]

print(f"Train episodes : {len(train_episodes)}  ×  {len(train_episodes[0])} steps")
print(f"Test  episodes : {len(test_episodes)}  ×  {len(test_episodes[0])} steps")
print(f"Workload range : [{min(train_series):.3f}, {max(train_series):.3f}]")

In [ ]:
# ── Visualise the first training episode's workload ──────────────────────────
fig, ax = plt.subplots(figsize=(10, 2.5))
ax.plot(train_episodes[0], linewidth=0.9, color='steelblue')
ax.set_xlabel('Timestep (1 step = 60 s)')
ax.set_ylabel('Normalised workload')
ax.set_title('Workload trace — first training episode (288 steps ≈ 4.8 h)')
ax.set_xlim(0, EPISODE_LENGTH - 1)
plt.tight_layout()
plt.show()

In [ ]:
# ── Run one full episode with a random policy; print first 5 steps ───────────
rng = np.random.default_rng(42)
env = EnergySchedulingEnv(train_episodes[0], config=DEFAULT_ENV_CONFIG)
obs, info = env.reset()

print(f"{'Step':>4}  {'Action':>6}  {'Workload':>8}  {'Queue':>6}  "
      f"{'Battery':>7}  {'Energy':>6}  {'Latency':>7}  {'Miss':>4}  {'Reward':>8}")
print('-' * 72)

step_log = []
done = False
t = 0
while not done:
    action = int(rng.integers(0, 3))
    obs, reward, done, info = env.step(action)
    step_log.append({
        'workload': info['arrival_norm'],
        'queue':    info['queue'],
        'action':   action,
        'energy':   info['step_energy'],
        'reward':   info['step_reward'],
    })
    if t < 5:
        print(f"{t+1:>4}  {DEFAULT_ENV_CONFIG.action_names[action]:>6}  "
              f"{info['arrival_norm']:>8.3f}  {info['queue']:>6.3f}  "
              f"{info['battery']:>7.2f}  {info['step_energy']:>6.3f}  "
              f"{info['step_latency']:>7.3f}  {int(info['step_miss']):>4}  "
              f"{info['step_reward']:>8.4f}")
    t += 1

print(f"\n  ... (episode length = {t} steps)")
total_return = sum(s['reward'] for s in step_log)
print(f"  Total episode return (random policy): {total_return:.3f}")

## 4) Baseline Policies

We evaluate fixed heuristics on the same held-out test episodes to establish non-learning references.


In [ ]:
from baselines import (
    always_low_policy, always_medium_policy, always_high_policy,
    threshold_policy_factory, evaluate_policy,
)

baseline_policies = {
    'always_low':    always_low_policy,
    'always_medium': always_medium_policy,
    'always_high':   always_high_policy,
    'threshold(w=0.60)': threshold_policy_factory(workload_threshold=0.60),
    'threshold(w=0.80)': threshold_policy_factory(workload_threshold=0.80),
}

print(f"{'Policy':<22}  {'Avg Return':>10}  {'Energy':>7}  {'Latency':>8}  {'Miss Rate':>9}")
print('-' * 65)
baseline_results = {}
for name, policy in baseline_policies.items():
    m = evaluate_policy(test_episodes, policy, config=DEFAULT_ENV_CONFIG)
    baseline_results[name] = m
    print(f"{name:<22}  {m['avg_episode_return']:>10.3f}  "
          f"{m['avg_step_energy']:>7.4f}  {m['avg_step_latency']:>8.4f}  "
          f"{m['miss_rate']:>9.4f}")

best_baseline_name   = max(baseline_results, key=lambda k: baseline_results[k]['avg_episode_return'])
best_baseline_return = baseline_results[best_baseline_name]['avg_episode_return']
print(f"\n  → Best baseline: {best_baseline_name}  (avg return = {best_baseline_return:.3f})"
      f"\n    RL methods in Section 5 will be compared against this value.")

## 5) RL Training

We train tabular and linear-approximation Q-learning (single-seed run for curves),
then use saved multi-seed artifacts for final comparison.


In [ ]:
import time
from q_learning import QLearningConfig, train_tabular_q_learning

# ── Tabular Q-learning (300 epochs, single seed) ─────────────────────────────
tabular_cfg = QLearningConfig(
    num_epochs=300,
    alpha=0.15,
    gamma=0.98,
    epsilon_start=0.30,
    epsilon_end=0.02,
    epsilon_decay=0.995,
    eval_every=5,
    seed=42,
)

t0 = time.perf_counter()
tabular_result = train_tabular_q_learning(
    train_episodes=train_episodes,
    test_episodes=test_episodes,
    env_config=DEFAULT_ENV_CONFIG,
    algo_config=tabular_cfg,
)
tabular_time = time.perf_counter() - t0

print(f"Tabular Q-learning  —  training time: {tabular_time:.2f} s")
bm = tabular_result['best_test_metrics']
print(f"  Best checkpoint (epoch {int(tabular_result['best_epoch'])}):")
print(f"    avg_return = {bm['avg_episode_return']:.4f}")
print(f"    energy     = {bm['avg_step_energy']:.4f}")
print(f"    latency    = {bm['avg_step_latency']:.4f}")
print(f"    miss_rate  = {bm['miss_rate']:.4f}")

In [ ]:
from approx_q import ApproxQLearningConfig, train_linear_approx_q_learning

# ── Linear Approximation Q-learning (400 epochs, single seed) ────────────────
approx_cfg = ApproxQLearningConfig(
    num_epochs=400,
    alpha=0.02,
    gamma=0.98,
    epsilon_start=0.30,
    epsilon_end=0.02,
    epsilon_decay=0.997,
    eval_every=5,
    seed=42,
)

t0 = time.perf_counter()
approx_result = train_linear_approx_q_learning(
    train_episodes=train_episodes,
    test_episodes=test_episodes,
    env_config=DEFAULT_ENV_CONFIG,
    algo_config=approx_cfg,
)
approx_time = time.perf_counter() - t0

print(f"Linear Approx Q-learning  —  training time: {approx_time:.2f} s")
print(f"  Feature vector dimension: {int(approx_result['feature_dim'])}")
bm = approx_result['best_test_metrics']
print(f"  Best checkpoint (epoch {int(approx_result['best_epoch'])}):")
print(f"    avg_return = {bm['avg_episode_return']:.4f}")
print(f"    energy     = {bm['avg_step_energy']:.4f}")
print(f"    latency    = {bm['avg_step_latency']:.4f}")
print(f"    miss_rate  = {bm['miss_rate']:.4f}")

In [ ]:
# ── Plot learning curves for both methods ────────────────────────────────────
tab_hist  = tabular_result['history']
apx_hist  = approx_result['history']

tab_epochs  = [h['epoch']               for h in tab_hist]
tab_returns = [h['eval_test_avg_return'] for h in tab_hist]
apx_epochs  = [h['epoch']               for h in apx_hist]
apx_returns = [h['eval_test_avg_return'] for h in apx_hist]

# best baseline for reference line
best_baseline_return = max(v['avg_episode_return'] for v in baseline_results.values())

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(tab_epochs, tab_returns, label='Tabular Q-learning (300 epochs)',       color='steelblue',  linewidth=1.5)
ax.plot(apx_epochs, apx_returns, label='Linear Approx Q-learning (400 epochs)', color='darkorange', linewidth=1.5)
ax.axhline(best_baseline_return, linestyle='--', color='gray', linewidth=1.2,
           label=f'Best baseline ({best_baseline_return:.2f})')
ax.set_xlabel('Epoch')
ax.set_ylabel('Avg test return')
ax.set_title('Learning curves — full training run (seed=42)')
ax.legend()
plt.tight_layout()
plt.show()

print(f"\nTraining time  |  Tabular: {tabular_time:.1f} s   Approx: {approx_time:.1f} s")

# ── Single-seed comparison vs best baseline ───────────────────────────────────
tab_best = tabular_result['best_test_metrics']['avg_episode_return']
apx_best = approx_result['best_test_metrics']['avg_episode_return']

print(f"\n{'─'*55}")
print(f"  Single-seed result vs best baseline")
print(f"{'─'*55}")
print(f"  {'Method':<32}  {'Avg Return':>10}  {'Δ vs baseline':>13}")
print(f"  {'─'*32}  {'─'*10}  {'─'*13}")
best_bl_label = max(baseline_results, key=lambda k: baseline_results[k]['avg_episode_return'])
print(f"  {f'Best baseline ({best_bl_label})':<32}  {best_baseline_return:>10.4f}  {'(reference)':>13}")
print(f"  {'Tabular Q-learning':<32}  {tab_best:>10.4f}  {tab_best - best_baseline_return:>+13.4f}")
print(f"  {'Linear Approx Q-learning':<32}  {apx_best:>10.4f}  {apx_best - best_baseline_return:>+13.4f}")
print(f"{'─'*55}")

## 6) Saved Best Models

Reload best tabular Q-table and best linear weights from saved artifacts,
and re-evaluate on the test set for consistent reporting.


In [ ]:
from q_learning import evaluate_q_policy, greedy_action as tab_greedy
from approx_q import LinearQApproximator, evaluate_linear_q_policy

# ── Load pre-trained artifacts from full experiment ───────────────────────────
TAB_Q_PATH  = ROOT / 'results/tabular_q_learning_120/q_table_best.npy'
APX_W_PATH  = ROOT / 'results/approx_q_learning_120/weights_best.npy'

q_table_best  = np.load(TAB_Q_PATH)
weights_best  = np.load(APX_W_PATH)
approximator  = LinearQApproximator(DEFAULT_ENV_CONFIG)

print("Q-table shape :", q_table_best.shape,
      "  (workload_bins × queue_bins × battery_bins × actions)")
print("Weight vector  :", weights_best.shape,
      f"  ({approximator.base_dim} base features × {approximator.num_actions} actions)")

# ── Evaluate saved models on test episodes ────────────────────────────────────
tab_saved = evaluate_q_policy(q_table_best, test_episodes, env_config=DEFAULT_ENV_CONFIG)
apx_saved = evaluate_linear_q_policy(weights_best, approximator, test_episodes,
                                      env_config=DEFAULT_ENV_CONFIG)

print("\n  Saved best model evaluation on test set:")
print(f"  {'Method':<28}  {'Avg Return':>10}  {'Energy':>7}  {'Latency':>8}  {'Miss Rate':>9}")
print('  ' + '-' * 65)
for label, m in [('Tabular Q (saved best)', tab_saved), ('Approx Q (saved best)', apx_saved)]:
    print(f"  {label:<28}  {m['avg_episode_return']:>10.4f}  "
          f"{m['avg_step_energy']:>7.4f}  {m['avg_step_latency']:>8.4f}  "
          f"{m['miss_rate']:>9.4f}")

### Linear Approximation Feature Design (Concise)

`Q(s,a) = w_a^T phi(s)` with action-block encoding.

Base feature groups:
- Bias + continuous terms: workload, normalized queue, battery ratio
- One-hot bins: workload(5), queue(5), battery(5)
- Interaction terms: workload*queue, queue*(1-battery), workload*(1-battery)

Total dimension: `22` base features x `3` actions = `66`.


In [ ]:
# ── Visualise learned Q-table: greedy action heatmap across battery bins ─────
action_names = DEFAULT_ENV_CONFIG.action_names
n_w   = len(DEFAULT_ENV_CONFIG.workload_bins) - 1   # 5
n_q   = len(DEFAULT_ENV_CONFIG.queue_bins)    - 1   # 5
n_b   = len(DEFAULT_ENV_CONFIG.battery_bins)  - 1   # 5

fig, axes = plt.subplots(1, n_b, figsize=(13, 3), sharey=True)
fig.suptitle(
    'Tabular Q-table: greedy action by (workload_bin, queue_bin) for each battery_bin\n'
    '(0 = low battery → 4 = full battery)',
    y=1.04,
)

cmap = plt.cm.get_cmap('viridis', 3)
for b_bin in range(n_b):
    ax = axes[b_bin]
    grid = np.zeros((n_q, n_w), dtype=int)
    for w in range(n_w):
        for q in range(n_q):
            grid[q, w] = int(np.argmax(q_table_best[w, q, b_bin]))   # ← use b_bin
    im = ax.imshow(grid, vmin=0, vmax=2, cmap=cmap, aspect='auto', origin='lower')
    ax.set_title(f'batt_bin={b_bin}')
    ax.set_xlabel('workload_bin')
    if b_bin == 0:
        ax.set_ylabel('queue_bin')
    ax.set_xticks(range(n_w))
    ax.set_yticks(range(n_q))

cbar = fig.colorbar(im, ax=axes, ticks=[0, 1, 2], shrink=0.85)
cbar.ax.set_yticklabels(action_names)
plt.tight_layout()
plt.show()

## 7) Regenerate Final Artifacts

This cell regenerates publication-style figures/tables into `results/final_figures/` from saved JSON/CSV artifacts.


In [ ]:
cmd = [
    'python3', str(ROOT / 'scripts/generate_final_artifacts.py'),
    '--trace-path', str(ROOT / 'data/processed/workload_trace_120.csv'),
    '--phase3-json', str(ROOT / 'results/phase3_multiseed_120.json'),
    '--reward-json', str(ROOT / 'results/reward_sensitivity_120.json'),
    '--hyper-json', str(ROOT / 'results/hyperparam_sensitivity_120.json'),
    '--generalization-json', str(ROOT / 'results/generalization_check_120.json'),
    '--tabular-history-csv', str(ROOT / 'results/tabular_q_learning_120/history.csv'),
    '--approx-history-csv', str(ROOT / 'results/approx_q_learning_120/history.csv'),
    '--tabular-q-table', str(ROOT / 'results/tabular_q_learning_120/q_table_best.npy'),
    '--approx-weights', str(ROOT / 'results/approx_q_learning_120/weights_best.npy'),
    '--output-dir', str(ROOT / 'results/final_figures'),
]
res = subprocess.run(cmd, capture_output=True, text=True, check=True)
print(res.stdout)

## 8) Main Results (Default Reward Setting)

### Comparison guardrail

Section 8 uses the default reward weights `(1.0, 0.6, 2.0)`.
Section 9 changes reward weights for ablation; raw returns across different reward settings are **not** directly comparable.


In [ ]:
from IPython.display import Markdown, display as ipy_display

summary_md_path = ROOT / 'results/final_figures/final_summary_table.md'
ipy_display(Markdown(summary_md_path.read_text(encoding='utf-8')))

### Figures


In [ ]:
from IPython.display import Image, display

fig_dir = ROOT / 'results/final_figures'
figure_names = [
    'learning_curves_tabular_vs_approx.png',
    'baseline_vs_rl_return.png',
    'energy_latency_tradeoff.png',
    'reward_sensitivity_return.png',
    'hyperparam_sensitivity_return.png',
    'policy_action_frequency.png',
]
for name in figure_names:
    path = fig_dir / name
    print('\n', name)
    display(Image(filename=str(path)))

## 9) Risk-Controlled Parameter Analysis

### A) Reward-sensitivity comparability (setting-wise)

Use `results/reward_sensitivity_context_120.md`, where each reward setting is compared against `always_low` under the **same** objective.

| (alpha, beta, gamma) | Approx Return | AlwaysLow Return | Gap vs AlwaysLow |
|---|---:|---:|---:|
| (1.00, 0.80, 2.00) | -56.463 | -59.105 | +2.642 |
| (0.80, 0.60, 2.00) | -45.098 | -46.921 | +1.822 |
| (1.00, 0.60, 2.00) default | -56.027 | -57.289 | +1.262 |

This removes the `-56` vs `-45` ambiguity by using within-setting gaps.

### B) Deadline-miss sensitivity stress test

Use `results/deadline_stress_test_120.md` with stricter deadline thresholds:
- At threshold `2.5` (default), miss rates are near zero.
- At threshold `0.5`, `always_low` miss rate rises to `0.0249`, while both RL policies are `0.00058`.

So miss-rate is not degenerate; the default threshold is simply lenient for this trace.

### C) Hyperparameter sensitivity (approx)

Top settings remain close (`~ -56.02 +/- 0.11 to 0.14`), indicating stable performance in a reasonable hyperparameter neighborhood.


## 10) Expected vs. Actual (Short)

| Expectation | Observation |
|---|---|
| RL beats heuristics | Yes, but margin is modest (about 1.3 points vs best baseline) |
| Approx beats tabular | Yes, by a small return margin |
| Tabular trains faster | Yes, ~5x speed advantage |
| Multi-seed variance is low | Yes, std is small across seeds |

Interpretation: the workload is relatively easy (best baseline already strong),
so gains are real but incremental.


## 11) Challenges and Future Work (Short)

### Main challenges

- Discretization trade-off for tabular coverage vs state fidelity
- Reward-weight calibration for meaningful energy/latency behavior
- Limited test stress under default deadline threshold

### Planned extensions

- Multi-device coordination and richer constraints
- Harder/stress-test workload slices and stricter service targets
- Stronger policy classes for larger/continuous action/state spaces


## 12) Conclusion

### Main comparison (5 seeds, 120 shards, non-overlap test)

| Method | Avg Return (mean +/- std) | Energy | Latency | Miss Rate | Train Time (s) |
|---|---|---|---|---|---|
| Linear Approx Q-learning | -56.016 +/- 0.117 | 0.190 +/- 0.001 | 0.008 +/- 0.001 | 0.000 | 34.4 +/- 0.17 |
| Tabular Q-learning | -56.200 +/- 0.084 | 0.191 +/- 0.001 | 0.007 +/- 0.001 | 0.000 | 6.3 +/- 0.04 |
| Baseline: always_low | -57.289 +/- 0.000 | 0.180 +/- 0.000 | 0.032 +/- 0.000 | 0.000 | - |

Takeaways:
- Both RL methods outperform the best baseline under the default objective.
- Approx has slightly better return; tabular is much faster.
- Added risk-control analyses clarify comparability and deadline sensitivity.
